[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/03_Decision_Tree_and_Random_Forest_Wine.ipynb)

# Decision Trees & Random Forests — Wine Classification

**Problem type:** supervised multiclass classification (3 cultivars)

---

## How decision trees work

A **decision tree** recursively splits the training data on the feature that reduces
impurity the most (measured by Gini index or information gain).  At each internal node the algorithm asks a yes/no question: *"Is `alcohol` <= 13.05?"*  Samples that satisfy the condition go left; the rest go right.  This continues until every leaf is pure (only one class) or a stopping criterion is hit (`max_depth`, `min_samples_leaf`, …).

**Key insight:** an unconstrained single tree can perfectly memorise the training set (100 % train accuracy) yet generalise poorly — this is **overfitting**.

## How random forests reduce overfitting

A **random forest** is an ensemble of many decision trees trained with two sources of
randomness:

1. **Bagging** – each tree sees a *bootstrap* sample (random rows with replacement).
2. **Feature sub-sampling** – at every split only √*p* features are considered.

At prediction time all trees **vote** and the majority class wins.  Because the trees are decorrelated, their individual errors cancel out and the ensemble generalises far better than any single tree.

---

## Dataset: `sklearn.datasets.load_wine`

| Property | Value |
|---|---|
| Samples | 178 |
| Features | 13 chemical-analysis measurements |
| Classes | 3 wine cultivars (0, 1, 2) |
| Source | UCI Wine Recognition Data Set |

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # non-interactive backend (works everywhere)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

np.random.seed(42)
print('All libraries loaded.')

In [ ]:
# ── Load the Wine dataset ─────────────────────────────────────────────
wine = load_wine(as_frame=True)

X = wine.data          # DataFrame, 178 × 13
y = wine.target        # Series, values 0 / 1 / 2
feature_names = wine.feature_names
target_names  = wine.target_names  # ['class_0', 'class_1', 'class_2']

print(f'Shape: {X.shape}')
print(f'\nClass distribution:\n{y.value_counts().sort_index()}')
print(f'\nClass names: {list(target_names)}')
print(f'\nFeature names:\n{list(feature_names)}')
X.head()

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Plot 1: class balance bar chart ───────────────────────────────────
counts = y.value_counts().sort_index()
axes[0].bar(target_names, counts, color=['#4C72B0', '#DD8452', '#55A868'],
            edgecolor='black')
axes[0].set_title('Class Balance')
axes[0].set_xlabel('Cultivar')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# ── Plot 2: alcohol distribution by class ─────────────────────────────
colors = ['#4C72B0', '#DD8452', '#55A868']
for cls, color in zip([0, 1, 2], colors):
    mask = y == cls
    axes[1].hist(X.loc[mask, 'alcohol'], bins=12, alpha=0.6,
                 label=target_names[cls], color=color, edgecolor='white')
axes[1].set_title('Alcohol Content by Class')
axes[1].set_xlabel('Alcohol')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# ── Plot 3: flavanoids distribution by class ───────────────────────────
for cls, color in zip([0, 1, 2], colors):
    mask = y == cls
    axes[2].hist(X.loc[mask, 'flavanoids'], bins=12, alpha=0.6,
                 label=target_names[cls], color=color, edgecolor='white')
axes[2].set_title('Flavanoids by Class')
axes[2].set_xlabel('Flavanoids')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.suptitle('Wine Dataset — EDA', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/_eda1.png', bbox_inches='tight')
plt.close()
print('EDA plot 1 saved.')

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────
corr = X.corr()

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

n = len(feature_names)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(feature_names, fontsize=8)

for i in range(n):
    for j in range(n):
        val = corr.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=6, color='black' if abs(val) < 0.7 else 'white')

ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/_corr.png', bbox_inches='tight')
plt.close()
print('Correlation heatmap saved.')

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train size : {X_train.shape[0]} samples')
print(f'Test  size : {X_test.shape[0]} samples')
print(f'\nTrain class counts:\n{y_train.value_counts().sort_index()}')
print(f'\nTest  class counts:\n{y_test.value_counts().sort_index()}')

In [ ]:
# ── Decision Tree: deep (unconstrained) vs shallow (max_depth=3) ──────

# 1. Deep tree — tends to overfit
dt_deep = DecisionTreeClassifier(random_state=42)
dt_deep.fit(X_train, y_train)
dt_deep_train_acc = accuracy_score(y_train, dt_deep.predict(X_train))
dt_deep_test_acc  = accuracy_score(y_test,  dt_deep.predict(X_test))

# 2. Shallow tree — generalises better at the cost of some bias
dt_shallow = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_shallow.fit(X_train, y_train)
dt_shallow_train_acc = accuracy_score(y_train, dt_shallow.predict(X_train))
dt_shallow_test_acc  = accuracy_score(y_test,  dt_shallow.predict(X_test))

print('=== Decision Tree Results ===')
print(f'  Deep   tree — train acc: {dt_deep_train_acc:.4f} | test acc: {dt_deep_test_acc:.4f}')
print(f'  Shallow tree — train acc: {dt_shallow_train_acc:.4f} | test acc: {dt_shallow_test_acc:.4f}')
print(f'\n  Deep tree depth : {dt_deep.get_depth()}')
print(f'  Shallow tree depth : {dt_shallow.get_depth()}')

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc  = accuracy_score(y_test,  rf.predict(X_test))

print('=== Random Forest Results ===')
print(f'  Train accuracy : {rf_train_acc:.4f}')
print(f'  Test  accuracy : {rf_test_acc:.4f}')

print('\n=== Summary comparison ===')
rows = [
    ['Deep Decision Tree',    dt_deep_train_acc,    dt_deep_test_acc],
    ['Shallow Decision Tree', dt_shallow_train_acc, dt_shallow_test_acc],
    ['Random Forest (200)',   rf_train_acc,         rf_test_acc],
]
print(f'{"Model":<25} {"Train Acc":>10} {"Test Acc":>10}')
print('-' * 47)
for name, tr, te in rows:
    print(f'{name:<25} {tr:>10.4f} {te:>10.4f}')

In [ ]:
# ── Confusion Matrix (Random Forest) ─────────────────────────────────
y_pred_rf = rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(target_names, fontsize=10)
ax.set_yticklabels(target_names, fontsize=10)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix — Random Forest', fontsize=12, fontweight='bold')

for i in range(3):
    for j in range(3):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=14, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('/tmp/_cm.png', bbox_inches='tight')
plt.close()
print('Confusion matrix saved.')

print('\n=== Classification Report (Random Forest) ===')
print(classification_report(y_test, y_pred_rf, target_names=target_names))

In [ ]:
# ── Visualise the Shallow Decision Tree (max_depth=3) ────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_shallow,
    feature_names=list(feature_names),
    class_names=list(target_names),
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
)
ax.set_title('Shallow Decision Tree (max_depth=3) — Wine Classification',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/_tree.png', bbox_inches='tight', dpi=120)
plt.close()
print('Shallow tree plot saved.')

In [ ]:
# ── Random Forest Feature Importances ────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(importances.index, importances.values,
               color='steelblue', edgecolor='black')

# Highlight top 3
top3_idx = importances.values.argsort()[-3:]
for idx in top3_idx:
    bars[idx].set_color('#DD8452')

ax.set_xlabel('Mean Decrease in Impurity (importance)', fontsize=11)
ax.set_title('Random Forest — Feature Importances\n(orange = top 3)',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
plt.tight_layout()
plt.savefig('/tmp/_fi.png', bbox_inches='tight')
plt.close()

print('Top 5 features by importance:')
print(importances.sort_values(ascending=False).head(5).to_string())

## Findings

| Model | Train Acc | Test Acc |
|---|---|---|
| Deep Decision Tree (unconstrained) | ~100 % | lower |
| Shallow Decision Tree (max_depth=3) | lower | comparable |
| Random Forest (200 trees) | ~100 % | **~97 – 100 %** |

**Overfitting in the single deep tree**  
The unconstrained decision tree achieves 100 % training accuracy by memorising every
sample, but its test accuracy drops — a clear sign of overfitting.  Constraining it to `max_depth=3` reduces variance at the cost of a little bias.

**Random forest generalises**  
With 200 trees, bagging and feature sub-sampling decorrelate the individual learners.
Their average vote irons out individual errors and the forest achieves near-perfect
test accuracy despite each tree still being deep.

**Most important chemical features**  
The feature-importance chart typically shows:
- **`flavanoids`** — polyphenol class that differs strongly across cultivars
- **`proline`** — amino-acid proxy for terroir
- **`color_intensity`** — direct visual marker
- **`od280/od315_of_diluted_wines`** — protein content proxy

Together these four features carry the majority of the discriminative signal.

## Try it yourself

### 1 · Bias–variance trade-off with `max_depth`
```python
depths = range(1, 15)
train_accs, test_accs = [], []
for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  clf.predict(X_test)))

plt.plot(depths, train_accs, label='Train')
plt.plot(depths, test_accs,  label='Test')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Bias–Variance Trade-off'); plt.legend(); plt.show()
```
*What you'll see:* as depth increases, train accuracy rises to 100 % and test accuracy peaks then levels off — the classic bias-variance curve.

### 2 · Change `n_estimators`
```python
for n in [10, 50, 100, 200, 500]:
    rf_ = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_.fit(X_train, y_train)
    print(n, accuracy_score(y_test, rf_.predict(X_test)))
```
*More trees → more stable predictions, but diminishing returns beyond ~100.*

### 3 · Try `min_samples_leaf`
```python
for msl in [1, 2, 5, 10, 20]:
    rf_ = RandomForestClassifier(n_estimators=200, min_samples_leaf=msl, random_state=42)
    rf_.fit(X_train, y_train)
    print(msl, accuracy_score(y_test, rf_.predict(X_test)))
```
*Higher `min_samples_leaf` smooths the trees but may hurt accuracy on small datasets.*